## Part 1: MongoDB Questions

In [ ]:
## load json to mongodb
from pymongo import MongoClient
import json

client = MongoClient('mongodb://localhost:27017/')
db = client['Epstein']

# docs=json.load(open("./data/documents_sample.json",encoding="utf-8"))
# with open("./data/epstein.json","w") as f:
#           f.write(json.dumps(docs))

docs = json.load(open("epstein.json"))
db.drop_collection('docs')
collect = db['docs']
collect.insert_many(docs)

### Query MongoDB


In [ ]:
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['Epstein']
col = db['docs']

projection = {
    "_id": 0,
    "doc_id":1,
    "people": 1,
    "org":1,
    "doc_type":1,
    "date":1
    }

In [ ]:
### Sample Query 1:
### find all docs that has EPSTEIN mentioned and org is DOJ
name="EPSTEIN"
org="DOJ"
# query = {
#     'people': {
#         '$in': [name]
#     },
#     'organizations':{
#         '$in': [org]
#     }
# }

### Sample Query 2:
### find all docs that either has EPSTEIN mentioned or DOJ mentioned
name="EPSTEIN"
org="DOJ"
query = {
     '$or': [
        {
            'people': {
                '$in': [name]
            }
        },
        {
            'organizations': {
                '$in': [org]
            }
        }
    ]
}

projection = {
    "_id": 0,
    "doc_id":1,
    "people":1,
    "organizations":1
    }
for doc in collect.find(query, projection):
    print( doc["doc_id"])
    # for i,p in enumerate(doc["people"]):
    #       if name in p:
    #            print(doc["people"][i], end=",")
    # for i,p in enumerate(doc["organizations"]):
    #     if org in p:
    #         print(doc["organizations"][i])
           

In [ ]:
## Sample Query 3
## sort names of people that are most frquently mentioned in descending orer of reqeuncy
people={}

for person_array, doc_id in map(
    lambda x: (x['people'],x['doc_id']),
    col.find(query, projection)
):
    for name in person_array:
        people[name] = people.get(name, 0) + 1

sorted (people.items(), key=lambda x:x[1], reverse=True)

---
## Part 2 : Relational Database Question


A sample of the data in a json file is shown below:
```python
[
  {
    "doc_id": "109-1",
    "url": "https://epstein-docs.github.io/document/109-1/",
    "document_type": "Court Filing",
    "date": "2024_09_17",
    "people": ['Maxwell', 'CABRANES', 'WESLEY'],
    "organizations": ['US of Appeals', 'US Court','DOJ'],
    "summary": "This is a ...",
    "full_text": "The United States Court...."
  },
  {
    "doc_id": "109-2",
    "url": "https://epstein-docs.github.io/document/109-2/",
    :
    :
  }
]
```

1.  Given the json data file schema above,design a Normalised (3NF) database model


#### Schema overview
```mermaid
erDiagram
    document ||--o{ mention : "referenced by"
    entity ||--o{ mention : "tagged in"
    document {
        TEXT doc_id PK
        TEXT url
        TEXT date
        TEXT summary
        TEXT doc_type
        TEXT full_text
    }
    entity {
        INTEGER entity_id PK
        TEXT name
        TEXT entity_type
    }
    mention {
        INTEGER id PK
        TEXT doc_id FK
        INTEGER entity_id FK
    }
```

2. Write SQL queries to find all documents that has EPSTEIN mentioned and org is DOJ

```python
SELECT d.doc_id, e1.name, e2.name FROM document as d
INNER JOIN mention as m1 ON d.doc_id = m1.doc_id
INNER JOIN entity as e1 ON e1.entity_type = "person" AND m1.entity_id = e1.entity_id
INNER JOIN mention as m2 ON d.doc_id = m2.doc_id
INNER JOIN entity as e2 ON e2.entity_type = "organization" AND m2.entity_id = e2.entity_id
WHERE 
e1.name = 'EPSTEIN'
AND e2.name = 'DOJ'
```

### Practical Exercise: SQL 
1.  create a relational database using the schema above and import data from json


In [ ]:
import json

def load_json(json_file):
    return json.load(open(json_file))

In [ ]:
import sqlite3

def create_schema(db_file) -> None:
    conn = sqlite3.connect(db_file)
    conn.executescript(
        """
        DROP TABLE IF EXISTS mention;
        DROP TABLE IF EXISTS entity;
        DROP TABLE IF EXISTS document;

        CREATE TABLE document (
            doc_id TEXT PRIMARY KEY,
            url TEXT,
            date TEXT,
            summary TEXT,
            doc_type TEXT,
            full_text TEXT
        );

        CREATE TABLE entity (
            entity_id INTEGER PRIMARY KEY,
            name TEXT UNIQUE NOT NULL,
            entity_type TEXT
            
        );

        CREATE TABLE mention (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            doc_id TEXT NOT NULL,
            entity_id INTEGER NOT NULL,
            FOREIGN KEY (doc_id) REFERENCES document(doc_id),
            FOREIGN KEY (entity_id) REFERENCES entity(entity_id)
        );
        """
    )
    conn.close()

In [ ]:
import sqlite3
def load_data(json_data, db_file):
    conn = sqlite3.connect(db_file)
    for doc in json_data:
        doc_id=doc['doc_id']
        url = doc['url']
        date = doc['date']
        full_text = doc['full_text']
        summary = doc['summary']
        people = doc['people']
        orgs = doc['organizations']
        doc_type=doc['document_type']
        conn.execute('''INSERT INTO document (doc_id, url, date, summary, doc_type, full_text)VALUES(?,?,?,?,?,?)''', (doc_id, url, date,summary, doc_type, full_text))
        for person in people:
            # insert into entity table
            conn.execute('''INSERT OR IGNORE INTO entity (name, entity_type) VALUES (?, ?)''', (person, 'person'))
            # get entity_id
            entity_id = conn.execute('''SELECT entity_id FROM entity WHERE name = ?''', (person,)).fetchone()[0]
            # insert into mention table
            conn.execute('''INSERT INTO mention (doc_id, entity_id) VALUES (?, ?)''', (doc_id, entity_id))
        for org in orgs:
            # insert into entity table
            conn.execute('''INSERT OR IGNORE INTO entity (name, entity_type) VALUES (?, ?)''', (org, 'organization'))
            # get entity_id
            entity_id = conn.execute('''SELECT entity_id FROM entity WHERE name = ?''', (org,)).fetchone()[0]
            # insert into mention table
            conn.execute('''INSERT INTO mention (doc_id, entity_id) VALUES (?, ?)''', (doc_id, entity_id))

    conn.commit()
    
    

In [ ]:
create_schema("epstein.db")
load_data(load_json('epstein.json'), 'epstein.db')

2. SQL Query 

In [ ]:
## query
import sqlite3
conn = sqlite3.connect('epstein.db')
name = "EPSTEIN"
org_name = "DOJ"
# Query 1: EPSTEIN mentioned and org is DOJ
# query = '''
# SELECT d.doc_id, e1.name, e2.name FROM document as d
# INNER JOIN mention as m1 ON d.doc_id = m1.doc_id
# INNER JOIN entity as e1 ON e1.entity_type = "person" AND m1.entity_id = e1.entity_id
# INNER JOIN mention as m2 ON d.doc_id = m2.doc_id
# INNER JOIN entity as e2 ON e2.entity_type = "organization" AND m2.entity_id = e2.entity_id
# WHERE 
# e1.name = ?
# AND e2.name = ?
# '''

#Query 2: find all docs that either has EPSTEIN mentioned or DOJ mentioned
query = """
SELECT DISTINCT d.doc_id
FROM document AS d
JOIN mention AS m ON d.doc_id = m.doc_id
JOIN entity AS e ON m.entity_id = e.entity_id
WHERE (e.entity_type = 'person' AND e.name = ?)
    OR (e.entity_type = 'organization' AND e.name = ?)
"""


results = conn.execute(query, (name, org_name)).fetchall()
for row in results:
    print(row)

---
## Part 3: Data Structures

In [ ]:
import json

def load_json(json_file):
    return json.load(open(json_file))

json_dicts = load_json("epstein.json")

In [ ]:
class Doc:
    def __init__(self, doc):
        for key, value in doc.items():
            setattr(self, key, value)
    def __repr__(self):
        return f"{self.doc_id}"
    


In [ ]:
import json
class HT:
    size = 60
    @classmethod
    def _hash(cls, obj:dict):
        return hash(json.dumps(obj))%HT.size
    
    def __init__(self):
        self.array = [None for _ in range(HT.size)]

    def insert(self, obj:dict):
        doc = Doc(obj)
        index = HT._hash(obj)
        cur = index
        while True:
            if self.array[cur] == None:
                self.array[cur] = doc
                return
            else:
                print("collision at index", cur)
                cur = (cur+1)%HT.size
                if cur == index:
                    raise Exception("HT is full")
    
    



In [ ]:
## load hash table
ht = HT()
for d in json_dicts:
    ht.insert(d)

In [ ]:
## build lookup tables for people, organisations
people_lookup = {} # "name": [index to HT,]
org_lookup = {}
for d in json_dicts:
    for person in d["people"]:
        people_lookup[person] = people_lookup.setdefault(person,set()) | {HT._hash(d)}
    for org in d["organizations"]:
        org_lookup[org] = org_lookup.setdefault(org,set()) | {HT._hash(d)}

In [80]:
## Query for all docs that has EPSTEIN mentioned and org is DOJ
for index in org_lookup["DOJ"] & people_lookup["EPSTEIN"]:
    print(ht.array[index])


# ## Query for all docs that has EPSTEIN mentioned or org is DOJ
# for index in org_lookup["DOJ"] | people_lookup["EPSTEIN"]:
#     print(ht.array[index])


120
2
6
6-1
